# OTT Subscriber Churn Analysis — Phase 1: Data Pipeline <span style="float:right; font-size: 16px; font-weight: normal; margin-top: 15px;">Author: Girish Pathak</span>

In [1]:
"""
=============================================================================
OTT Subscriber Churn Analysis — Phase 1: Data Pipeline
=============================================================================
Project     : ott-subscriber-churn-analysis
Author      : [Girish Pathak]
Description : Reads raw Excel data → cleans each table → engineers features
              → loads into SQLite database → exports master DataFrame as CSV

Tech Stack  : Python, Pandas, NumPy, SQLite3
Dataset     : customer_churn_data_raw.xlsx (3 sheets)
Output      : customer_churn_clean.db  (SQLite database)
              master_churn_data.csv    (analysis-ready flat file)
=============================================================================
"""
import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime

## Step 1a: Configuration Variables

In [2]:
TODAY = pd.Timestamp(datetime.today().date())

EXCEL_PATH  = "customer_churn_data_raw.xlsx"
DB_PATH     = "customer_churn_clean.db"
CSV_OUTPUT  = "master_churn_data.csv"

# Reference date for age and tenure calculations
TODAY = pd.Timestamp(datetime.today().date())

# File paths — update these if your folder structure is different
EXCEL_PATH  = "customer_churn_data_raw.xlsx"
DB_PATH     = "customer_churn_clean.db"
CSV_OUTPUT  = "master_churn_data.csv"

print("=" * 60)
print("  OTT Churn Analysis — Phase 1: Data Pipeline")
print("=" * 60)

  OTT Churn Analysis — Phase 1: Data Pipeline


## Step 1b: Load Raw Data from Excel

In [3]:
xl = pd.ExcelFile("customer_churn_data_raw.xlsx")

df_customer = pd.read_excel(xl, sheet_name="db_customer")
df_sub      = pd.read_excel(xl, sheet_name="db_subscription")
df_support  = pd.read_excel(xl, sheet_name="db_support")

# Verify
print("df_customer columns:", df_customer.columns.tolist())
print("Shape:", df_customer.shape)

df_customer columns: ['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']
Shape: (21, 8)


## Step 2: Cleaning db_customer


In [4]:
# ── Step 2a: Drop columns with 100% or near-100% null values (useless for analysis)
# pincode  → 100% null  (No values)
# interests → 81% null  (It is useless for analysis)
df_customer = df_customer.drop(columns=["pincode", "interests"])
print("  ✓ Dropped 'pincode' (100% null) and 'interests' (81% null)")
print("  Remaining columns:", df_customer.columns.tolist())

  ✓ Dropped 'pincode' (100% null) and 'interests' (81% null)
  Remaining columns: ['customerid', 'name', 'country', 'state', 'gender', 'dob']


In [5]:
# ── Step 2b: Standardize gender values — 4 variants → 2 clean values
# Problem: There are 4 variants — Male, Men, Female, Women
# Solution: Convert in 2 clean values using Dictionary mapping

gender_map = {
    "Male"  : "Male",
    "Men"   : "Male",     # "Men" → "Male"
    "Female": "Female",
    "Women" : "Female"    # "Women" → "Female"
}

df_customer = df_customer.assign(
    gender = df_customer["gender"].map(gender_map)
)

print("\n  ✓ Gender standardized")
print("  Unique values now:", df_customer["gender"].unique().tolist())


  ✓ Gender standardized
  Unique values now: ['Male', 'Female']


In [6]:
# ── Step 2c: Fill missing country values with 'Unknown'
# Country is missing in 3 rows
# Replace it with "Unknown" using fillna

df_customer = df_customer.assign(
    country = df_customer["country"].fillna("Unknown")
)

print("\n  ✓ Country nulls filled with 'Unknown'")
print("  Null count now:", df_customer["country"].isnull().sum())


  ✓ Country nulls filled with 'Unknown'
  Null count now: 0


In [7]:
# ── Step 2d: Convert dob to datetime format for age calculation
# DOB is imported from Excel as a string/object
# Convert it to datetime before calculating age.

df_customer = df_customer.assign(
    dob = pd.to_datetime(df_customer["dob"])
)

print("\n  ✓ DOB converted to datetime")
print("  Sample:", df_customer["dob"].head(3).tolist())


  ✓ DOB converted to datetime
  Sample: [Timestamp('1982-04-12 00:00:00'), Timestamp('1995-11-23 00:00:00'), Timestamp('1978-02-15 00:00:00')]


In [8]:
# ── Step 2e: Feature Engineering — Create the AGE column 
# The 'dob' column is not directly useful for analysis.
# Create an 'age' column to compare different age groups.
# Formula: (Current Date - DOB) / 365 = Age in years

TODAY = pd.Timestamp("today").normalize()

df_customer["age"] = (
    (TODAY - df_customer["dob"]).dt.days // 365
).astype(int)

print("\n  ✓ Feature Engineered: 'age' column")
print("  Age range:", df_customer["age"].min(), "to", df_customer["age"].max())


  ✓ Feature Engineered: 'age' column
  Age range: 21 to 49


## STEP 3: db_subscription Cleaning

In [9]:
# First, a quick look at the raw data
print("\n  Raw df_sub info:")
print(f"  Shape: {df_sub.shape}")
print(f"  Columns: {df_sub.columns.tolist()}")
print(f"\n  Null counts:\n{df_sub.isnull().sum()}")


  Raw df_sub info:
  Shape: (21, 11)
  Columns: ['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

  Null counts:
customerid                  0
subscription_start_date     0
subscription_type           0
renewal_date                0
plan_type                   0
contract_type               0
cancellation_date          15
cancellation_reason        15
monthly_charges             0
cltv                        0
churn_score                 0
dtype: int64


In [10]:
# ── Step 3a: Fix the typos
# 'Refferal' is a spelling mistake and should be corrected to 'Referral'.
# replace() changes only the specified value while leaving all other values unchanged.
# rest values are untouched

df_sub = df_sub.assign(
    subscription_type = df_sub["subscription_type"].replace(
        "Refferal", "Referral"
    )
)

print("\n  ✓ Typo fixed: 'Refferal' → 'Referral'")
print("  Unique values:", df_sub["subscription_type"].unique().tolist())


  ✓ Typo fixed: 'Refferal' → 'Referral'
  Unique values: ['Referral', 'Paid', 'Organic']


In [11]:
# ── Step 3b: Convert date columns to datetime
# Dates imported from Excel are often stored as strings.
# Convert them to datetime format for date-based calculations.
#
# IMPORTANT: 'cancellation_date' contains 15 null values (active customers).
# errors='coerce' → converts null or invalid values to NaT (Not a Time), preventing the script from crashing.

df_sub = df_sub.assign(
    subscription_start_date = pd.to_datetime(
        df_sub["subscription_start_date"]
    ),
    cancellation_date = pd.to_datetime(
        df_sub["cancellation_date"], errors="coerce"
    )
)

print("\n  ✓ Date columns converted to datetime")
print("  subscription_start_date dtype:", df_sub["subscription_start_date"].dtype)
print("  cancellation_date dtype      :", df_sub["cancellation_date"].dtype)
print("  cancellation_date nulls      :", df_sub["cancellation_date"].isnull().sum(),
      "(these are ACTIVE customers — expected!)")


  ✓ Date columns converted to datetime
  subscription_start_date dtype: datetime64[ns]
  cancellation_date dtype      : datetime64[ns]
  cancellation_date nulls      : 15 (these are ACTIVE customers — expected!)


In [12]:
# ── Step 3c: Feature Engineering — Create the CHURN FLAG
# This is the TARGET VARIABLE for our analysis.
#
# Logic:
#   If 'cancellation_date' exists → Customer has churned → 1
#   If 'cancellation_date' is null → Customer is active → 0
#
# .notna() returns True or False.
# .astype(int) converts True to 1 and False to 0.

df_sub["churn_flag"] = df_sub["cancellation_date"].notna().astype(int)

print("\n  ✓ Feature Engineered: 'churn_flag'")
print("  Distribution:")
print(df_sub["churn_flag"].value_counts().rename({0:"Active",1:"Churned"}))


  ✓ Feature Engineered: 'churn_flag'
  Distribution:
churn_flag
Active     15
Churned     6
Name: count, dtype: int64


In [13]:
# ── Step 3d: Feature Engineering — Create TENURE MONTHS
# Business question: "Do new customers churn more than long-term customers?"
# The 'subscription_start_date' alone cannot answer this question.
# Creating a 'tenure_months' column makes the analysis possible.
#
# Formula: (Current Date - Subscription Start Date) / 30 = Tenure in Months

df_sub["tenure_months"] = (
    (TODAY - df_sub["subscription_start_date"]).dt.days // 30
).astype(int)

print("\n  ✓ Feature Engineered: 'tenure_months'")
print("  Range:", df_sub["tenure_months"].min(),
      "to", df_sub["tenure_months"].max(), "months")
print("  Average tenure:", df_sub["tenure_months"].mean().round(1), "months")


  ✓ Feature Engineered: 'tenure_months'
  Range: 34 to 87 months
  Average tenure: 56.3 months


In [14]:
# ── Step 3e: Feature Engineering — Create REVENUE LOST
# Business KPI: How much monthly revenue is lost due to customer churn?
#
# Formula: churn_flag × monthly_charges
#   Active customer  (flag = 0): 0 × 12.99 = 0.00   (no revenue loss)
#   Churned customer (flag = 1): 1 × 12.99 = 12.99  (revenue lost)

df_sub["revenue_lost"] = df_sub["churn_flag"] * df_sub["monthly_charges"]

print("\n  ✓ Feature Engineered: 'revenue_lost'")
print("  Total monthly revenue lost: ₹",
      df_sub["revenue_lost"].sum().round(2))
print("  Per churned customer avg  : ₹",
      df_sub[df_sub["churn_flag"]==1]["revenue_lost"].mean().round(2))


  ✓ Feature Engineered: 'revenue_lost'
  Total monthly revenue lost: ₹ 73.94
  Per churned customer avg  : ₹ 12.32


In [15]:
# ── Step 3f: Final Check
print("\n" + "="*45)
print("  STEP 3 COMPLETE — df_sub Summary")
print("="*45)
print(f"  Shape   : {df_sub.shape}")
print(f"  Columns : {df_sub.columns.tolist()}")
print(f"\n  Null values:\n{df_sub.isnull().sum()}")
print("\nSample data (churned customers only):")
df_sub[df_sub["churn_flag"]==1][
    ["customerid","plan_type","churn_flag",
     "tenure_months","revenue_lost"]
]


  STEP 3 COMPLETE — df_sub Summary
  Shape   : (21, 14)
  Columns : ['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score', 'churn_flag', 'tenure_months', 'revenue_lost']

  Null values:
customerid                  0
subscription_start_date     0
subscription_type           0
renewal_date                0
plan_type                   0
contract_type               0
cancellation_date          15
cancellation_reason        15
monthly_charges             0
cltv                        0
churn_score                 0
churn_flag                  0
tenure_months               0
revenue_lost                0
dtype: int64

Sample data (churned customers only):


,customerid,plan_type,churn_flag,tenure_months,revenue_lost
1,0003-MKNFE,Premium,1,72,12.99
4,0013-EXCHZ,Standard,1,43,13.99
6,0013-SMEOE,Basic,1,58,8.99
11,0017-IUDMW,Standard,1,40,7.99
13,0019-EFAEP,Basic,1,47,12.99
18,0022-TCJCI,Basic,1,34,16.99


## STEP 4: db_support Cleaning


In [16]:
# Quick Look

xl = pd.ExcelFile("customer_churn_data_raw.xlsx")
df_support_check = pd.read_excel(xl, sheet_name="db_support")

print("Columns:", df_support_check.columns.tolist())
print("\nFull data:\n")
print(df_support_check.to_string())

Columns: ['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']

Full data:

   customerid complaint_date escalations  csat_score  col_1            comment
0  0003-MKNFE     2024-08-28           N          60    NaN      service issue
1  0003-MKNFE     2024-08-28           Y          10    NaN     demaned refund
2  0013-EXCHZ     2024-01-20           Y          20    NaN                NaN
3  0013-MHZWF     2025-03-18           N          90    NaN  guidance to renew
4  0013-SMEOE     2024-11-01           N          30    NaN                NaN
5  0017-IUDMW     2024-04-10           Y          25    NaN                NaN
6  0019-EFAEP     2024-09-27           Y          30    NaN                NaN
7  0022-TCJCI     2024-09-13           Y          10    NaN                NaN
8  0022-TCJCI     2024-09-14           N          90    NaN    received refund


In [17]:
# ── Step 4a: Drop unnecessary columns
# 'col_1' contains 100% null values and has no business value. So drop col_1

df_support = df_support_check.drop(columns=["col_1"])

print("Columns after drop:", df_support.columns.tolist())
print("\nEscalations unique values:", df_support["escalations"].unique())

Columns after drop: ['customerid', 'complaint_date', 'escalations', 'csat_score', 'comment']

Escalations unique values: ['N' 'Y']


In [18]:
print("  ✓ col_1 already dropped")
print("  Columns:", df_support.columns.tolist())

  ✓ col_1 already dropped
  Columns: ['customerid', 'complaint_date', 'escalations', 'csat_score', 'comment']


In [20]:
# ── Step 4b: Fill missing values in the 'comment' column
# The 'comment' column has 5 missing values.
# Leaving nulls in a text column can lead to silent data loss during groupby operations.
# Fill missing values with the placeholder "No comment" to keep the data complete and trackable.

df_support = df_support.assign(
    comment = df_support["comment"].fillna("No comment")
)
print("\n  ✓ Comment nulls filled")
print("  Null count:", df_support["comment"].isnull().sum())


  ✓ Comment nulls filled
  Null count: 0


In [21]:
# ── Step 4c: Encode 'escalations' from Y/N to 1/0
# 'Y' and 'N' are text values, so mathematical operations cannot be performed on them.
# Convert them to 1 and 0 so that you can:
#   - calculate the average (escalation rate)
#   - calculate the total (number of escalations)
#   - analyze the correlation with customer churn

df_support = df_support.assign(
    escalations = df_support["escalations"].map({"Y": 1, "N": 0})
)

print("\n  ✓ Escalations encoded: Y→1, N→0")
print("  Unique values now:", df_support["escalations"].unique().tolist())
print("  Total escalated tickets:", df_support["escalations"].sum())


  ✓ Escalations encoded: Y→1, N→0
  Unique values now: [0, 1]
  Total escalated tickets: 5


In [22]:
# ── Step 4d: Convert 'complaint_date' to datetime
# Convert the date from string format to datetime for date-based calculations.

df_support = df_support.assign(
    complaint_date = pd.to_datetime(
        df_support["complaint_date"], errors="coerce"
    )
)

print("\n  ✓ complaint_date converted to datetime")
print("  Dtype:", df_support["complaint_date"].dtype)


  ✓ complaint_date converted to datetime
  Dtype: datetime64[ns]


In [24]:
# ── Step 4e: Feature Engineering — AGGREGATION
# PROBLEM: A single customer can have multiple support tickets.
#          Joining the data directly would create duplicate customer rows.
#
# SOLUTION: Group by 'customerid' so that each customer has only one row.
#
# Create three new features for each customer:
#   total_complaints → Total number of support tickets raised
#   has_escalation   → Whether any ticket was escalated (max = 1 if at least one escalation occurred)
#   avg_csat         → Average customer satisfaction (CSAT) score

df_support_agg = df_support.groupby("customerid").agg(
    total_complaints = ("csat_score",   "count"),  # ticket count
    has_escalation   = ("escalations",  "max"),    # any escalation?
    avg_csat         = ("csat_score",   "mean")    # avg satisfaction
).reset_index()
# .reset_index() → 'customer id' becomes a regular column again.

# round avg_csat
df_support_agg["avg_csat"] = df_support_agg["avg_csat"].round(1)

print("\n  ✓ Feature Engineered: Aggregated support per customer")
print("  Before aggregation:", df_support.shape[0], "rows (tickets)")
print("  After aggregation :", df_support_agg.shape[0], "rows (customers)")
print("\n  Aggregated data:")
print(df_support_agg)


  ✓ Feature Engineered: Aggregated support per customer
  Before aggregation: 9 rows (tickets)
  After aggregation : 7 rows (customers)

  Aggregated data:
   customerid  total_complaints  has_escalation  avg_csat
0  0003-MKNFE                 2               1      35.0
1  0013-EXCHZ                 1               1      20.0
2  0013-MHZWF                 1               0      90.0
3  0013-SMEOE                 1               0      30.0
4  0017-IUDMW                 1               1      25.0
5  0019-EFAEP                 1               1      30.0
6  0022-TCJCI                 2               1      50.0


In [25]:
# ── Step 4 Final Check 
print(f"\n{'='*45}")
print(f"  STEP 4 COMPLETE")
print(f"{'='*45}")
print(f"  Null check:\n{df_support_agg.isnull().sum()}")


  STEP 4 COMPLETE
  Null check:
customerid          0
total_complaints    0
has_escalation      0
avg_csat            0
dtype: int64


## STEP 5: MASTER DATAFRAME (3-TABLE JOIN)

In [32]:
# ── Join 1: Customer + Subscription
# LEFT JOIN — Retain all 21 customers, even if some have no matching subscription records.

df_master = df_customer.merge(
    df_sub,
    on  = "customerid",
    how = "left"
)
print(f"  ✓ Step 1 Join done: customer + subscription")
print(f"    Shape: {df_master.shape}")

  ✓ Step 1 Join done: customer + subscription
    Shape: (21, 20)


In [33]:
# ── Join 2: Add Support Data
# LEFT JOIN — Customers with no matching support records
# will have NaN values (these will be handled in the next step).

df_master = df_master.merge(
    df_support_agg,
    on  = "customerid",
    how = "left"
)
print(f"\n  ✓ Step 2 Join done: + support")
print(f"    Shape: {df_master.shape}")


  ✓ Step 2 Join done: + support
    Shape: (21, 23)


In [36]:
# ── Handle missing values in support-related columns
# Customers who never contacted support
# will have missing values. Fill them with 0.

df_master["total_complaints"] = (
    df_master["total_complaints"].fillna(0).astype("Int64")
)
df_master["has_escalation"] = (
    df_master["has_escalation"].fillna(0).astype("Int64")
)
df_master["avg_csat"] = df_master["avg_csat"].fillna(0)

In [37]:
# ── New Feature: has_complaint 
# Binary flag indicating whether a customer has ever filed a complaint.
# 0 = Never filed a complaint
# 1 = Filed at least one complaint

df_master["has_complaint"] = (
    (df_master["total_complaints"] > 0).astype("Int64")
)

print(f"\n  ✓ NaN values fixed")
print(f"  ✓ Feature Engineered: 'has_complaint'")


  ✓ NaN values fixed
  ✓ Feature Engineered: 'has_complaint'


In [38]:
# ── Final Check

print(f"\n{'='*45}")
print(f"  STEP 5 COMPLETE — Master DataFrame")
print(f"{'='*45}")
print(f"  Shape   : {df_master.shape}")
print(f"  Columns : {df_master.columns.tolist()}")
print(f"\n  Null check:")
print(df_master.isnull().sum()[df_master.isnull().sum() > 0])
print(f"\n  Churn breakdown:")
print(df_master["churn_flag"].value_counts().rename({0:"Active", 1:"Churned"}))
print(f"\nSample (churned customers):")
df_master[df_master["churn_flag"]==1][
    ["customerid","plan_type","churn_flag",
     "revenue_lost","tenure_months","has_complaint"]
]


  STEP 5 COMPLETE — Master DataFrame
  Shape   : (21, 24)
  Columns : ['customerid', 'name', 'country', 'state', 'gender', 'dob', 'age', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score', 'churn_flag', 'tenure_months', 'revenue_lost', 'total_complaints', 'has_escalation', 'avg_csat', 'has_complaint']

  Null check:
cancellation_date      15
cancellation_reason    15
dtype: int64

  Churn breakdown:
churn_flag
Active     15
Churned     6
Name: count, dtype: int64

Sample (churned customers):


,customerid,plan_type,churn_flag,revenue_lost,tenure_months,has_complaint
1,0003-MKNFE,Premium,1,12.99,72,1
4,0013-EXCHZ,Standard,1,13.99,43,1
6,0013-SMEOE,Basic,1,8.99,58,1
11,0017-IUDMW,Standard,1,7.99,40,1
13,0019-EFAEP,Basic,1,12.99,47,1
18,0022-TCJCI,Basic,1,16.99,34,1


## STEP 6: Save the Data to an SQLite Database

In [39]:
# ── Step 6a: Create a database connection.

import sqlite3

# Create a database connection.
# If the database file already exists, it will be opened.
# Otherwise, a new database file will be created.

conn = sqlite3.connect("customer_churn_clean.db")

# ── Save the cleaned tables 
# if_exists="replace" → Replace the existing table with the new one.
# index=False → Do not save the Pandas row index as a column.

df_customer.to_sql(
    "db_customer", conn,
    if_exists="replace", index=False
)
print("  ✓ db_customer saved")

df_sub.to_sql(
    "db_subscription", conn,
    if_exists="replace", index=False
)
print("  ✓ db_subscription saved")

df_support_agg.to_sql(
    "db_support", conn,
    if_exists="replace", index=False
)
print("  ✓ db_support saved")

  ✓ db_customer saved
  ✓ db_subscription saved
  ✓ db_support saved


In [40]:
# ──  Step 6b: Save Master table
df_master.to_sql(
    "master_churn", conn,
    if_exists="replace", index=False
)
print("  ✓ master_churn saved")

# commit + close
conn.commit()
conn.close()

print(f"\n{'='*45}")
print(f"  STEP 6 COMPLETE — Database Ready")
print(f"{'='*45}")
print(f"  File saved: 'customer_churn_clean.db'")
print(f"  Tables inside:")
print(f"    → db_customer     ({df_customer.shape[0]} rows)")
print(f"    → db_subscription ({df_sub.shape[0]} rows)")
print(f"    → db_support      ({df_support_agg.shape[0]} rows)")
print(f"    → master_churn    ({df_master.shape[0]} rows)")

  ✓ master_churn saved

  STEP 6 COMPLETE — Database Ready
  File saved: 'customer_churn_clean.db'
  Tables inside:
    → db_customer     (21 rows)
    → db_subscription (21 rows)
    → db_support      (7 rows)
    → master_churn    (21 rows)


## STEP 7: Export CSV

In [41]:
df_master.to_csv("master_churn_data.csv", index=False)

print(f"  ✓ Saved: 'master_churn_data.csv'")
print(f"  Rows   : {df_master.shape[0]}")
print(f"  Columns: {df_master.shape[1]}")

  ✓ Saved: 'master_churn_data.csv'
  Rows   : 21
  Columns: 24


## STEP 8: Final Summery

In [44]:
# ============================================================
# PIPELINE SUMMARY REPORT
# ============================================================
total_customers = len(df_master)
churned         = int(df_master["churn_flag"].sum())
active          = total_customers - churned
churn_rate      = round((churned / total_customers) * 100, 1)
total_rev_lost  = df_master["revenue_lost"].sum()
avg_tenure      = df_master["tenure_months"].mean().round(1)
complainers     = int(df_master["has_complaint"].sum())

print(f"""
{'='*50}
  PIPELINE SUMMARY REPORT
{'='*50}
  Total Customers        : {total_customers}
  Churned                : {churned}  ({churn_rate}%)
  Active                 : {active}  ({100-churn_rate}%)
  Monthly Revenue Lost   : ₹{total_rev_lost:.2f}
  Avg Customer Tenure    : {avg_tenure} months
  Customers w/ Complaints: {complainers}
{'─'*50}
  Output Files:
    → customer_churn_clean.db
    → master_churn_data.csv
{'─'*50}
  Features Engineered:
    ✓ age
    ✓ churn_flag       (target variable)
    ✓ tenure_months
    ✓ revenue_lost     (business KPI)
    ✓ total_complaints
    ✓ has_escalation
    ✓ avg_csat
    ✓ has_complaint
{'='*50}

✅ Phase 1 COMPLETED!
   Next is Phase 2 — EDA & Visualization
""")


  PIPELINE SUMMARY REPORT
  Total Customers        : 21
  Churned                : 6  (28.6%)
  Active                 : 15  (71.4%)
  Monthly Revenue Lost   : ₹73.94
  Avg Customer Tenure    : 56.3 months
  Customers w/ Complaints: 7
──────────────────────────────────────────────────
  Output Files:
    → customer_churn_clean.db
    → master_churn_data.csv
──────────────────────────────────────────────────
  Features Engineered:
    ✓ age
    ✓ churn_flag       (target variable)
    ✓ tenure_months
    ✓ revenue_lost     (business KPI)
    ✓ total_complaints
    ✓ has_escalation
    ✓ avg_csat
    ✓ has_complaint

✅ Phase 1 COMPLETED!
   Next is Phase 2 — EDA & Visualization

